# Cookie Cats: dời gate từ level 30 sang 40 có làm giảm retention không?

Cookie Cats là game xếp hình. Ở một level nhất định người chơi gặp **gate**: phải chờ hoặc trả tiền
để đi tiếp. Nhà phát hành chia ngẫu nhiên người chơi mới: `gate_30` (control, gate ở level 30) và
`gate_40` (treatment, gate ở level 40).

Cột: `userid`, `version`, `sum_gamerounds` (số ván chơi trong 14 ngày đầu), `retention_1`,
`retention_7` (có quay lại chơi sau 1 / 7 ngày không).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (có dòng chú thích `# explore` ở đầu) là chỗ đọc dữ liệu thật, vẽ, gọi hàm.
  Test bỏ qua các cell này.
- Chấm: `uv run pytest tracks/data-science/modules/03-ab-testing/exercises/intermediate/01-cookie-cats-retention -q`

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py cookie-cats)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(ROOT / "data/raw/cookie-cats/cookie_cats.csv")
df.head()

## 1. Hiểu dữ liệu trước khi kiểm định

Trước khi tính p-value nào: mỗi userid xuất hiện mấy lần? Có giá trị lạ không? Hai nhóm có cỡ
tương đương thiết kế 50/50 không (dùng hàm SRM của bài Basic 03)?

In [ ]:
# explore: số dòng, userid trùng, phân phối sum_gamerounds
print(len(df), df.userid.nunique())
df.sum_gamerounds.describe()

In [ ]:
# explore: có một người chơi 49 854 ván trong 14 ngày. Lỗi log hay bot? Ảnh hưởng gì tới metric retention?
import matplotlib.pyplot as plt

df[df.sum_gamerounds < 200].sum_gamerounds.hist(bins=100)
plt.title("sum_gamerounds (cắt ở 200)")
plt.show()

## 2. Bảng tóm tắt theo nhóm

In [ ]:
def retention_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Index là version; cột n (số người chơi), retention_1, retention_7 (tỉ lệ)."""
    # TODO: groupby version
    raise NotImplementedError

In [ ]:
# explore
retention_summary(df)

## 3. Khoảng tin cậy bằng bootstrap

Ước lượng phân phối của `mean(gate_40) − mean(gate_30)` bằng cách lấy mẫu lại có hoàn lại **trong
từng nhóm**. Khoảng tin cậy là percentile `alpha/2` và `1 − alpha/2` của các chênh lệch bootstrap.

Bootstrap 2 000 lần trên 45 nghìn dòng mỗi nhóm có thể chậm nếu làm ngây thơ. Hãy nghĩ: với cột
0/1, tổng của một mẫu bootstrap có phân phối gì?

In [ ]:
def bootstrap_diff_ci(df: pd.DataFrame, metric: str, n_boot: int = 2000, seed: int = 0,
                      alpha: float = 0.05) -> tuple[float, float, float]:
    """Trả (diff, lo, hi): diff = mean(gate_40) − mean(gate_30) trên dữ liệu gốc, [lo, hi] là CI bootstrap."""
    # TODO: rng = np.random.default_rng(seed)
    # TODO: lặp n_boot lần (hoặc vector hoá): resample từng nhóm, tính chênh lệch trung bình
    raise NotImplementedError

In [ ]:
# explore
for m in ["retention_1", "retention_7"]:
    print(m, bootstrap_diff_ci(df, m))

## 4. Ra quyết định

Quy tắc (viết trước khi xem kết quả): dựa trên CI 95% bootstrap của metric chính
- CI nằm hoàn toàn trên 0 → `"ship_gate_40"`
- CI nằm hoàn toàn dưới 0 → `"keep_gate_30"`
- CI chứa 0 → `"inconclusive"`

In [ ]:
def decision(df: pd.DataFrame, metric: str = "retention_7") -> str:
    # TODO: dùng bootstrap_diff_ci với tham số mặc định
    raise NotImplementedError

In [ ]:
# explore
decision(df)

## 5. Viết khuyến nghị

Viết 5–8 câu cho nhà phát hành: kết quả, mức độ chắc chắn, cơ chế có thể giải thích, những gì
thí nghiệm **không** đo (doanh thu in-app, retention dài hơn), và vấn đề SRM (p ≈ 0.0086) cần
xử lý thế nào.